In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e11/sample_submission.csv
/kaggle/input/playground-series-s5e11/train.csv
/kaggle/input/playground-series-s5e11/test.csv


## 1. Import Libraries 

In [29]:
#for data preprocessing 
import pandas as pd
import numpy as np

#for data processing (specific to ML modelling)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

#for training 
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


#for eval
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score 

## 2. Reading in file 

In [30]:
df = pd.read_csv("/kaggle/input/playground-series-s5e11/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s5e11/test.csv")
df_sub = pd.read_csv("/kaggle/input/playground-series-s5e11/sample_submission.csv")

## 3. Exploratory Data Analysis 

In [31]:
#this looks at first 5 rows 
df.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [32]:
df_test.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade
0,593994,28781.05,0.049,626,11461.42,14.73,Female,Single,High School,Employed,Other,D5
1,593995,46626.39,0.093,732,15492.25,12.85,Female,Married,Master's,Employed,Other,C1
2,593996,54954.89,0.367,611,3796.41,13.29,Male,Single,Bachelor's,Employed,Debt consolidation,D1
3,593997,25644.63,0.110,671,6574.30,9.57,Female,Single,Bachelor's,Employed,Debt consolidation,C3
4,593998,25169.64,0.081,688,17696.89,12.80,Female,Married,PhD,Employed,Business,C1


In [33]:
#checking null values for training data 
df.isnull().sum()

id                      0
annual_income           0
debt_to_income_ratio    0
credit_score            0
loan_amount             0
interest_rate           0
gender                  0
marital_status          0
education_level         0
employment_status       0
loan_purpose            0
grade_subgrade          0
loan_paid_back          0
dtype: int64

In [34]:
#checking null values for testing data 
df_test.isnull().sum()

id                      0
annual_income           0
debt_to_income_ratio    0
credit_score            0
loan_amount             0
interest_rate           0
gender                  0
marital_status          0
education_level         0
employment_status       0
loan_purpose            0
grade_subgrade          0
dtype: int64

## 4. Data Preprocessing 

### This includes cleaning the data, assigning X and y etc, encoding categorical variables 

In [35]:
#for training data
X = df.drop(columns = ["loan_paid_back"])
y = df["loan_paid_back"]

In [36]:
#for test data 
X_test_final = df_test

In [37]:
#retrieving categorical columns and numerical columns 

numerical_cols =  df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()


print(f"categorical columns : {categorical_cols}")
print(f"numerical columns : {numerical_cols}")

categorical columns : ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
numerical columns : ['id', 'annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'loan_paid_back']


In [38]:
#apply label encoding consistently to both train and test
for col in categorical_cols:
    le = LabelEncoder()
    #fit on combined data to ensure all categories are seen
    combined = pd.concat([X[col], X_test_final[col]], axis=0)
    le.fit(combined)
    X[col] = le.transform(X[col])
    X_test_final[col] = le.transform(X_test_final[col])

Some revision

pd.concat = combines 2 dataframes together (concatenates them)
le.fit = 
le.transform = 

In [39]:
X.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade
0,0,29367.99,0.084,736,2528.42,13.67,0,2,1,2,6,12
1,1,22108.02,0.166,636,4593.10,12.92,1,1,2,0,2,17
2,2,49566.20,0.097,694,17005.15,9.76,1,2,1,0,2,14
3,3,46858.25,0.065,533,4682.48,16.10,0,2,1,0,2,25
4,4,25496.70,0.053,665,12184.43,10.21,1,1,1,0,6,15


In [40]:
X_test_final.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade
0,593994,28781.05,0.049,626,11461.42,14.73,0,2,1,0,6,19
1,593995,46626.39,0.093,732,15492.25,12.85,0,1,2,0,6,10
2,593996,54954.89,0.367,611,3796.41,13.29,1,2,0,0,2,15
3,593997,25644.63,0.110,671,6574.30,9.57,0,2,0,0,2,12
4,593998,25169.64,0.081,688,17696.89,12.80,0,1,4,0,0,10


In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

We split the training data into train and validation sets to check how well the model generalizes before testing on the final test set.

Train set → used to fit the model

Validation set → used to tune hyperparameters and detect overfitting

Analogy: practise test before final exam/test

## 5. Training the data + Making the predictions 

In [42]:
# Initialize base model
xgb_clf = XGBClassifier(
    objective='binary:logistic',  # because we want probability of loan repayment
    eval_metric='auc',            # AUC is a common metric to evaluate performance
    use_label_encoder=False,      # avoid XGBoost's label encoder (as we have already done it)
    random_state=42,              # makes results reproducible
    n_jobs=-1                     # use all CPU cores to speed it up (1 = use 1 core -> slower processing) 
)


In [43]:
# Defining search space
param_dist = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}


Above are some very common parameters in tree-based models like XGboost, LightGBM and Gradient Boosting. 

* n_estimators → Number of trees (or boosting rounds). More trees = model can learn more patterns but risk of overfitting.

* max_depth → Maximum depth of each tree. Controls model complexity (deeper trees capture more patterns but can overfit).

* learning_rate → How quickly the model learns. (Or more specifically it controls how much each new tree contributes to correcting the erros of previous trees) 

* subsample → Fraction of training data used for each tree. Adds randomness to reduce overfitting (e.g., 0.8 = use 80% of data per tree).

There are many more hyperparameters. However with more hyperparameters, the training and tuning process takes much longer!

In [44]:
opt = RandomizedSearchCV(          #this is a random search
    estimator=xgb_clf,              #which model to tune 
    param_distributions=param_dist, # what hyperparameter options to try
    n_iter=10,                     # how many random combinations to test
    scoring='roc_auc',             # how to measure performance
    cv=3,                          # how many folds for cross validation (We will look at this in coming weeks)
    verbose=1,                     # how much progress info to show  (0= no messages, 1 = minimal messages, 2 = detailed messages...)
    random_state=42,               # fixes randomness so results are reproducible 
    n_jobs=-1                      # use all CPU cores to speed it up (1 = use 1 core -> slower processing) 
)

#fit the model/ training it 
opt.fit(X_train, y_train)

print("Best parameters:", opt.best_params_)
print("Best ROC-AUC:", opt.best_score_)

# Pick the best model and return predicted probabilities 
best_model = opt.best_estimator_
y_pred_proba = best_model.predict_proba(X_test)[:, 1]  

# Evaluate on test set
print("Test ROC-AUC:", roc_auc_score(y_test, y_pred_proba))


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.2}
Best ROC-AUC: 0.9195048792336938
Test ROC-AUC: 0.9219362694557477


#### Interpreting ROC/AUC scores
* 1.0 = Perfect model
* 0.5 = random guess ( not great)
* < 0.5 = worse than random

#### BUT WE HAVE TO MAKE FINAL PREDICTIONS ON THE GIVEN TEST DATA TO SUBMIT!

## 6. Final Predictions and Submission + Check!

In [45]:
y_pred_final = best_model.predict_proba(X_test_final)[:, 1]
# Show first few predictions
print("Sample predictions for X_test:")
print(y_pred_final[:10])

Sample predictions for X_test:
[0.9646204  0.9784203  0.47192582 0.9398113  0.9600721  0.9884496
 0.99227023 0.9808363  0.88481826 0.00582298]


In [46]:
submission = pd.DataFrame({'id': df_sub['id'],'loan_paid_back': y_pred_final})
submission

# Save to CSV
submission.to_csv('submission.csv', index=False)

In [24]:
submission.head()

,id,loan_paid_back
0,593994,0.964620
1,593995,0.978420
2,593996,0.471926
3,593997,0.939811
4,593998,0.960072


In [25]:
submission.shape

(254569, 2)

In [26]:
df_sub.shape

(254569, 2)

### You can make this model better! Try different models like LGBM and catboost, different hyperparameters and try to beat this score!